# Exercises XP Gold: Titanic. Student Notebook

This notebook is guided to complete the execices on the plateforme.
For each exercise, the **Instructions** are guided and some parts are already coded, and the **Guidance** explains exactly what you must do in complete sentences.

All **Learning Point** sections thoughout this notebook will provide you important concepts and insights to retain.

## What will you create
- A comprehensive machine learning project focused on predicting the survival of passengers on the Titanic.
- A series of robust classification models trained on the Titanic dataset.
- Detailed comparison reports of model performances with and without hyperparameter tuning.
- Customized hyperparameter tuning processes for different types of models.
- Visualizations of data insights, model performances, and feature importances.
- A final, well-documented Jupyter Notebook showcasing the entire analysis process.

## What will you learn
- Skills in data preparation, including dealing with missing values and feature engineering specifically for the Titanic dataset.
- Understanding of model-specific data preprocessing requirements.
- Experience in building, evaluating, and tuning various classification models like Decision Trees, K-Nearest Neighbors (KNN), and Neural Networks.
- Advanced techniques in hyperparameter tuning using Grid Search and Randomized Search.
- Deep insights into model evaluation metrics specific to classification tasks like accuracy, precision, recall, and F1 score.
- Development of a systematic approach to compare different machine learning models and select the best one for a given problem.
- For the below exercises, you will use the titanic dataset

## Dataset
If a Titanic CSV file is available in your working directory or `./data` (for example, `titanic.csv` with a `Survived` column), you will load it.
If it is not available, a synthetic Titanic-like dataset will be generated so you can complete the exercises.

### Learning point
You improve faster by making small, clear steps. Load the data, clean it, split it, and only then try models. Keep the target separate from features to avoid leaking answers.

## Exercise 1: Exploratory Data Analysis

### Instructions
Load the Titanic dataset.
Perform data cleaning and handle missing values.
Conduct exploratory data analysis to understand the relationships between different features and survival.

### Learning point
EDA is about asking simple questions first. How many people survived? Does class or gender seem related to survival? Simple counts and plots tell useful stories.

In [2]:
# TODO: Load the Titanic dataset into a DataFrame named df.
# In complete sentences inside your own comments, identify the target column as 'Survived' (0 or 1).
# If the dataset uses other column names, rename them to a standard set when possible.
# Print df.shape, df.dtypes.head(), total missing values, and df['Survived'].value_counts().to_dict().

import pandas as pd
from pathlib import Path
import zipfile
import os

# Unzip the dataset file if it exists
zip_file_path = '/content/Titanic Dataset.zip'
csv_file_name = 'Titanic-Dataset.csv' # Corrected the filename here
extract_dir = '/content/'

if Path(zip_file_path).exists():
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print(f"'{csv_file_name}' extracted from '{zip_file_path}' to '{extract_dir}'.")

# Load the Titanic dataset into a DataFrame named df.
# The target column is 'Survived' (0 or 1).
df = pd.read_csv(os.path.join(extract_dir, csv_file_name))

# Print df.shape
print(f"DataFrame shape: {df.shape}")

# Print df.dtypes.head()
print("\nDataFrame dtypes (first 5):\n", df.dtypes.head())

# Print total missing values
print("\nTotal missing values per column:\n", df.isnull().sum())

# Print df['Survived'].value_counts().to_dict()
print("\nValue counts for 'Survived' column:\n", df['Survived'].value_counts().to_dict())

'Titanic-Dataset.csv' extracted from '/content/Titanic Dataset.zip' to '/content/'.
DataFrame shape: (891, 12)

DataFrame dtypes (first 5):
 PassengerId     int64
Survived        int64
Pclass          int64
Name           object
Sex            object
dtype: object

Total missing values per column:
 PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

Value counts for 'Survived' column:
 {0: 549, 1: 342}


In [ ]:

# This piece of code is already prefilled, run it to execute it and see the results.
# It creates a synthetic Titanic-like dataset if you did not load 'df' above.

import pandas as pd
import numpy as np

try:
    df  # will raise NameError if not defined
    print("Dataset already loaded.")
except NameError:
    rng = np.random.default_rng(42)
    n = 1000
    df = pd.DataFrame({
        "Pclass": rng.integers(1, 4, size=n),
        "Sex": rng.choice(["male","female"], size=n),
        "Age": np.clip(rng.normal(30, 14, size=n), 0, None),
        "SibSp": rng.integers(0, 5, size=n),
        "Parch": rng.integers(0, 4, size=n),
        "Fare": np.abs(rng.normal(32, 50, size=n)),
        "Embarked": rng.choice(["S","C","Q"], size=n, p=[0.7,0.2,0.1]),
    })
    # Simple survival rule + noise to mimic correlations
    base = 0.3 + 0.15*(df["Sex"]=="female") + 0.1*(df["Pclass"]==1) - 0.1*(df["Pclass"]==3) + 0.05*(df["Fare"]>40)
    df["Survived"] = (rng.random(n) < base.clip(0,1)).astype(int)
    # Add some missingness
    miss_idx = rng.choice(n, size=int(0.1*n), replace=False)
    df.loc[miss_idx, "Age"] = np.nan
    print("Synthetic Titanic-like dataset created. Shape:", df.shape)
    print(df.head(3))

In [3]:
# TODO: Create a cleaned DataFrame named df_clean.
# Fill missing numeric values (like Age) with the median and missing categorical values (like Embarked) with the most frequent value.
# Optionally create a simple 'FamilySize' feature as SibSp + Parch + 1 and keep it in df_clean.
# Print the number of remaining missing values in df_clean.

import pandas as pd

df_clean = df.copy()

# Fill missing numeric values (like Age) with the median
# The 'Age' column has 177 missing values according to the previous output.
median_age = df_clean['Age'].median()
df_clean['Age'].fillna(median_age, inplace=True)

# Fill missing categorical values (like Embarked) with the most frequent value.
# The 'Embarked' column has 2 missing values.
# The 'Cabin' column has a large number of missing values (687), and for this exercise, we will drop it since it's hard to impute meaningfully without further analysis.

# Drop Cabin column due to excessive missing values for simplicity in this step.
df_clean.drop('Cabin', axis=1, inplace=True)

# Fill Embarked missing values with the most frequent value (mode).
mode_embarked = df_clean['Embarked'].mode()[0]
df_clean['Embarked'].fillna(mode_embarked, inplace=True)

# Optionally create a simple 'FamilySize' feature as SibSp + Parch + 1 and keep it in df_clean.
df_clean['FamilySize'] = df_clean['SibSp'] + df_clean['Parch'] + 1

# Print the number of remaining missing values in df_clean.
print("Number of remaining missing values in df_clean after cleaning:")
print(df_clean.isnull().sum())

Number of remaining missing values in df_clean after cleaning:
PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
FamilySize     0
dtype: int64


/tmp/ipykernel_13813/693268070.py:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_clean['Age'].fillna(median_age, inplace=True)
/tmp/ipykernel_13813/693268070.py:24: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', tr

In [ ]:

# This piece of code is already prefilled, run it to execute it and see the results.
# It shows simple plots for survival counts, survival by Pclass, and an Age histogram.

import matplotlib.pyplot as plt

surv_counts = df_clean["Survived"].value_counts().sort_index()
plt.figure()
surv_counts.plot(kind="bar")
plt.title("Survival counts (0=No, 1=Yes)")
plt.xlabel("Survived")
plt.ylabel("Count")
plt.show()

pclass_surv = df_clean.groupby("Pclass")["Survived"].mean()
plt.figure()
pclass_surv.plot(kind="bar")
plt.title("Mean survival rate by Pclass")
plt.xlabel("Pclass")
plt.ylabel("Mean survival")
plt.show()

plt.figure()
df_clean["Age"].hist(bins=20)
plt.title("Age distribution")
plt.xlabel("Age")
plt.ylabel("Count")
plt.show()

In [ ]:

# TODO: Split df_clean into features X and target y ('Survived'), then perform a stratified train-test split with test_size=0.2 and random_state=42.
# Name the outputs X_train, X_test, y_train, y_test and print their shapes in a complete sentence.

from sklearn.model_selection import train_test_split

raise NotImplementedError("Please create X, y, perform stratified split, and print shapes.")

### Learning point
Models read numbers. Use a preprocessing step to convert text columns to numbers and to fill any missing values. Pipelines keep preprocessing attached to the model so you do not leak information.

In [ ]:

# This piece of code is already prefilled, run it to execute it and see the results.
# It defines a helper to build a preprocessing transformer for numeric and categorical columns.

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

def make_preprocessor(num_cols, cat_cols, scale_numeric=True):
    num_steps = []
    num_steps.append(("imputer", SimpleImputer(strategy="median")))
    if scale_numeric:
        num_steps.append(("scaler", StandardScaler()))
    num_pipe = Pipeline(steps=num_steps)

    cat_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])

    pre = ColumnTransformer(transformers=[
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ])
    return pre

## Exercise 2: Decision Tree Classifier without Grid Search

### Instructions
Implement a Decision Tree Classifier on the Titanic dataset.
Manually choose and set the hyperparameters.
Evaluate its performance using accuracy, precision, recall, and F1 score.

### Learning point
Decision trees work on raw numbers and one-hot encoded categories. Deeper trees can overfit. Start simple and grow only if needed.

In [ ]:

# This piece of code is already prefilled, run it to execute it and see the results.
# It defines a function to print metrics and draw a confusion matrix.

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

def report_classification(y_true, y_pred):
    print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
    print("Precision:", round(precision_score(y_true, y_pred, zero_division=0), 4))
    print("Recall:", round(recall_score(y_true, y_pred, zero_division=0), 4))
    print("F1 score:", round(f1_score(y_true, y_pred, zero_division=0), 4))
    ConfusionMatrixDisplay.from_predictions(y_true, y_pred)
    plt.title("Confusion Matrix")
    plt.show()

In [ ]:

# TODO: Choose numeric and categorical columns from X_train and build a preprocessing transformer with make_preprocessor.
# TODO: Create a Pipeline that includes the preprocessor and a DecisionTreeClassifier with your chosen hyperparameters.
# TODO: Fit the pipeline, predict on X_test, and call report_classification.

from sklearn.tree import DecisionTreeClassifier

raise NotImplementedError("Please build the Decision Tree pipeline, fit it, and evaluate it.")

## Exercise 3: Decision Tree Classifier with Grid Search

### Instructions
Apply GridSearchCV to find the optimal hyperparameters for the Decision Tree Classifier.
Compare its performance with the manually tuned model from Exercise 2.

### Learning point
Grid search tries a small grid of settings and keeps the best. Limit the grid so you learn fast and avoid long runs.

In [ ]:

# TODO: Build a Pipeline with the same preprocessor and a DecisionTreeClassifier.
# TODO: Define a small parameter grid over max_depth, min_samples_split, and min_samples_leaf.
# TODO: Run GridSearchCV with cv=5. Print best_params_ and best_score_. Evaluate best_estimator_ on the test set.

from sklearn.model_selection import GridSearchCV

raise NotImplementedError("Please configure and run GridSearchCV for the Decision Tree.")

## Exercise 4: K-Nearest Neighbors (KNN) without Grid Search

### Instructions
Train a KNN classifier on the Titanic dataset without using hyperparameter tuning.
Choose the number of neighbors and distance metric based on your understanding.
Assess the model’s performance and discuss the choice of hyperparameters.

### Learning point
KNN is simple. It predicts by voting among nearby points. Scale numeric features or distances will be dominated by large-scale columns.

In [ ]:

# TODO: Build a Pipeline with make_preprocessor (with numeric scaling turned on) and KNeighborsClassifier.
# Choose n_neighbors and metric. Fit on the training set, predict on the test set, and report metrics.

from sklearn.neighbors import KNeighborsClassifier

raise NotImplementedError("Please build and evaluate the KNN pipeline without grid search.")

## Exercise 5: K-Nearest Neighbors (KNN) with Grid Search

### Instructions
Use GridSearchCV to optimize the hyperparameters of the KNN classifier, like the number of neighbors and distance metric.
Evaluate and compare the performance of the tuned model against the model from Exercise 4.

### Learning point
Try a few k values. Too small can be noisy, too large can wash out local patterns. Cross-validation helps you choose.

In [ ]:

# TODO: Build a Pipeline with the preprocessor and KNeighborsClassifier.
# Define a parameter grid for n_neighbors and metric (e.g., ['minkowski','manhattan']).
# Run GridSearchCV with cv=5. Print best_params_ and best_score_. Evaluate best_estimator_ on X_test.

raise NotImplementedError("Please configure and run GridSearchCV for KNN and compare to Exercise 4.")

## Exercise 6: Neural Network Classifier without Hyperparameter Tuning

### Instructions
Build a basic Neural Network to classify Titanic passengers.
Set the layers, neurons, and activation functions based on your intuition.
Analyze the model’s effectiveness in predicting survival.

### Learning point
Neural networks need scaled inputs and enough data. Start small. A compact model often learns faster and avoids overfitting.

In [ ]:

# TODO: Build a Pipeline with make_preprocessor (scaling on) and MLPClassifier.
# Choose hidden_layer_sizes, activation, alpha, and max_iter. Fit on the training data, predict on the test data, and report metrics.
# If you prefer TensorFlow or PyTorch, note in a comment how you would replace MLPClassifier with your chosen library.

from sklearn.neural_network import MLPClassifier

raise NotImplementedError("Please build and evaluate the MLP-based neural network pipeline.")

## Exercise 7 (Optional): Neural Network Classifier with Hyperparameter Tuning

### Instructions
Implement a Neural Network and use techniques like RandomizedSearchCV (or other applicable methods) to tune hyperparameters like the number of layers, neurons, and learning rate.
Evaluate the performance and compare it with the Neural Network from Exercise 6.

### Learning point
Tune a few knobs at a time. Use randomized search for a quick scan. Keep iterations small so you see results sooner.

In [ ]:

# TODO: Wrap your MLP pipeline in RandomizedSearchCV with a small param distribution over hidden_layer_sizes, alpha, and learning_rate_init.
# Use cv=3 and a modest n_iter. Print best_params_ and best_score_. Evaluate the best_estimator_ on the test set.
# If using TensorFlow in Colab, explain briefly how you would connect KerasClassifier to RandomizedSearchCV.

from sklearn.model_selection import RandomizedSearchCV
import numpy as np

raise NotImplementedError("Please configure and run RandomizedSearchCV for the MLP pipeline and report results.")